## Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.feature_selection import SelectFromModel
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import seaborn as sns
from scipy.stats import uniform, randin 

## Lasso Functions

### Create LagFeatureTransformer to add lag to Dataset

In [ ]:
class LagFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_lag = 93):
        self.max_lag = max_lag

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = pd.DataFrame(X, columns = ['y'])
        for lag in range(1, self.max_lag + 1):
            df[f'lag_{lag}'] = df['y'].shift(lag)
        return df.dropna().reset_index(drop=True)

### Create LassoFeatureSelector to select lag useful for prediction

In [ ]:
def apply_lasso_feature_selection(X, y, alpha_range = np.logspace(-5, 1, 100), top_n_features = 8):
    print("Running Lasso Feature Selection...")
    lasso = LassoCV(cv = TimeSeriesSplit(n_splits = 5), alphas = alpha_range, max_iter = 2000)
    lasso.fit(X, y)
    model = SelectFromModel(lasso, prefit = True, max_features = top_n_features)
    selected_mask = model.get_support()
    selected_features = X.columns[selected_mask]
    print(f"Selected {len(selected_features)} features: {list(selected_features)}")
    return selected_features


## Model Building and Data Handling

### Recursive Prediction Function

In [ ]:
def recursive_predict(model, X_test_initial, num_steps, selected_features):
    X_test_rec = X_test_initial.copy()
    predictions = []
    for _ in range(num_steps):
        y_pred = model.predict(X_test_rec[selected_features])
        predictions.append(y_pred[0])
        # shift lag features forward
        new_row = X_test_rec.iloc[0].shift(1)
        new_row.iloc[0] = y_pred[0]
        X_test_rec.iloc[0] = new_row
    return np.array(predictions)

### Parameter Grid

In [ ]:
param_distributions = {
    "LightGBM": {
        "num_leaves": [20, 31, 40, 50, 60],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "n_estimators": [50, 100, 150, 200],
        "min_child_samples": [5, 10, 20, 50],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "reg_alpha": [0, 0.1, 0.5, 1.0],
        "reg_lambda": [0, 0.1, 0.5, 1.0]
    },
    "XGBoost": {
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "n_estimators": [50, 100, 150, 200],
        "min_child_weight": [1, 3, 5, 10],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.3, 0.5],
        "reg_alpha": [0, 0.1, 0.5, 1.0],
        "reg_lambda": [0, 0.1, 0.5, 1.0]
    },
    "CatBoost": {
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
        "iterations": [50, 100, 150, 200],
        "l2_leaf_reg": [1, 3, 5, 10],
        "border_count": [32, 64, 128],
        "bagging_temperature": [0, 0.5, 1, 2],
        "random_strength": [1, 5, 10],
        "colsample_bylevel": [0.7, 0.8, 0.9, 1.0]
    }
}

### Define scoring using myscore

In [ ]:
def weighted_mae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)
weighted_mae_scorer = make_scorer(weighted_mae, greater_is_better=False)

### Train and Compare Models

In [ ]:
def evaluate_models(models, X, y, selected_features):
    train, test = train_test_split(X, test_size=0.2, shuffle=False)  # Ensures time order is maintained
    X_train, y_train = train.drop(columns=['y']), train['y']
    X_test, y_test = test.drop(columns=['y']), test['y']
    results = {}
    for model_name, model in models.items():
        print(f"Tuning {model_name} with RandomizedSearchCV...")
        search = RandomizedSearchCV(model, param_distributions[model_name], n_iter=50, scoring='weighted_mae_scorer', cv=TimeSeriesSplit(n_splits=5), n_jobs=-1)
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        print(f"Best parameters for {model_name}: {search.best_params_}")
        best_model.fit(X_train, y_train)
        num_steps = len(y_test)
        y_pred = recursive_predict(best_model, X_test.iloc[:-1], num_steps, selected_features)
        results[model_name] = weighted_mae(y_test, y_pred)
        print(f"{model_name} MAE: {results[model_name]:.4f}")
    return pd.DataFrame.from_dict(results, orient = 'index', columns = ['MAE'])



## Model Training

### Do Lag Transform and Selection

In [ ]:
print("Applying Lag Features...")
lag_transformer = LagFeatureTransformer(max_lag = 93)
X_lagged = lag_transformer.fit_transform(pd.DataFrame({'y': y}))
# lag selection
selected_features = apply_lasso_feature_selection(X_lagged.drop(columns=['y']), X_lagged['y'])
X_selected = X_lagged[['y'] + list(selected_features)]


### Do Manual Data Deletion Here

### Train Model

In [ ]:
results_df = evaluate_models({"LightGBM": lgb.LGBMRegressor(), "XGBoost": xgb.XGBRegressor(objective="reg:squarederror"), "CatBoost": cb.CatBoostRegressor(verbose=0)}, X_selected, X_selected['y'], selected_features)
print("Model Comparison:")
print(results_df)
sns.barplot(x=results_df.index, y=results_df['MAE'])
plt.title("Model Comparison (MAE)")
plt.show()
